In [ ]:
from pathlib import Path
import sys
import pandas as pd

REPO_ROOT = Path("..").resolve()
SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

import DIHS_Correlator as dc

In [ ]:
df = pd.read_csv('../data/processed/caio_italy_benchmark/full_italian_data.csv')
df = df.loc[:, ~df.columns.str.startswith('Unnamed:')]
df

In [ ]:
df.columns

In [ ]:
major_cols = ['SIO2N', 'TIO2N', 'AL2O3N', 'FE2O3TN', 'CAON', 'MGON', 'MNON', 'NA2ON', 'K2ON', 'P2O5N']
trace_cols = ['NbN', 'ZrN', 'LaN', 'CeN', 'SrN', 'BaN', 'RbN']
feature_spaces = {
    'coupled': {
        'columns': major_cols + trace_cols,
        'transform_type': 'clr',
        'major_cols': major_cols,
        'trace_cols': trace_cols,
    },
    'major_only': {
        'columns': major_cols,
        'transform_type': 'clr',
        'major_cols': major_cols,
        'trace_cols': [],
    },
    'trace_only': {
        'columns': trace_cols,
        'transform_type': 'scaled',
        'major_cols': [],
        'trace_cols': trace_cols,
    },
}

assert (df['lettercode'] == 'Caio').sum() == 17

all_results = {}
for feature_space_name, cfg in feature_spaces.items():
    output_dir = f'../results/2_caio_source_attribution/{feature_space_name}'
    exclude_columns = [
        c for c in df.columns
        if c != 'lettercode' and c not in cfg['columns']
    ]

    print(f"Running feature space: {feature_space_name} | transform={cfg['transform_type']}")
    result = dc.perturbative_triple_run_with_resolvedness(
        df=df,
        transform_type=cfg['transform_type'],
        unknown_sample='Caio',
        class_column='lettercode',
        random_state=12345,
        n_iterations=100,
        major_cols=cfg['major_cols'],
        trace_cols=cfg['trace_cols'],
        major_error=0.02,
        trace_error=0.10,
        perturbation_seed=20260108,
        compute_pairwise=True,
        plot_everything=True,
        write_files=True,
        output_dir=output_dir,
        plot_output_dir=output_dir,
        max_depth=100,
        exclude_columns=tuple(exclude_columns),
        pairwise_plot_order=None,
        save_cluster_data=True,
        save_untransformed=True,
        pseudo_unknown_iterations=100,
        pseudo_unknown_sample_size=17,
        pseudo_unknown_random_state=12345,
        target_precisions=[],
        min_runs_above_threshold=1,
        integration_depth=None,
        verbose=True,
        return_details=True,
    )
    all_results[feature_space_name] = result

all_results.keys()

In [ ]:
summary_frames = []
for feature_space_name, details in all_results.items():
    summary = details.get('summary')
    if summary is None or summary.empty:
        continue
    tagged = summary.copy()
    tagged.insert(0, 'feature_space', feature_space_name)
    summary_frames.append(tagged)

if summary_frames:
    combined_summary = pd.concat(summary_frames, ignore_index=True)
    combined_summary.to_csv('../results/2_caio_source_attribution/caio_resolvedness_summary_all_feature_spaces.csv', index=False)
    combined_summary
else:
    print('No resolvedness summary tables were produced.')